# Tools: from function to context

1. We run the tool loop manually to see the model's decision and the Python result.
2. `@tool` and `create_agent` execute the same loop.
3. `args_schema` provides the model with an explicit schema and validates its arguments.
4. `ToolRuntime` provides the tool with data from `context`, and `Command` saves the result in `state`.

[LangChain Tools Documentation](https://docs.langchain.com/oss/python/langchain/tools)


In [ ]:
# Imports and project path setup only; examples start in the next cell.
from dataclasses import dataclass, replace
from datetime import datetime
from inspect import getdoc, signature
from pathlib import Path
import json
import sys
from typing import Literal

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import AgentState, create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, messages_to_dict
from langgraph.types import Command
from model_config import get_json_model, get_model, get_model_name
from pydantic import BaseModel, Field, ValidationError

In [ ]:
model = get_model()
model_name = get_model_name()

### 1. Manual tool loop

These are ordinary Python functions. First we write their tool schemas by hand. Then we build the same schemas automatically from function names, type hints, and docstrings. `bind_tools` sends a schema to the model; our Python loop still executes the chosen function.

In [ ]:
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


TOOLS = {"add": add, "multiply": multiply}
MANUAL_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two integers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    },
]


def schema_from_function(function) -> dict:
    """Read this lesson's int arguments and description from a function."""
    properties = {}
    required = []
    for name, parameter in signature(function).parameters.items():
        if parameter.annotation is not int:
            raise TypeError(f"This example expects an int argument: {name}")
        properties[name] = {"type": "integer"}
        if parameter.default is parameter.empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": function.__name__,
            "description": getdoc(function),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


AUTO_TOOL_SCHEMAS = [schema_from_function(function) for function in TOOLS.values()]
print("Hand-written schema:")
print(json.dumps(MANUAL_TOOL_SCHEMAS, indent=2))
print("Schema read from function signatures and docstrings:")
print(json.dumps(AUTO_TOOL_SCHEMAS, indent=2))

TOOL_SCHEMAS = MANUAL_TOOL_SCHEMAS  # Change to AUTO_TOOL_SCHEMAS to compare.
model_with_tools = model.bind_tools(TOOL_SCHEMAS)
print("Available tools:", list(TOOLS))

The loop below shows every step: user → model decision → Python tool → tool result → model again. The model may ask for several tools in one step. We stop when it answers without tool calls. The next model call includes the earlier tool request and its corresponding tool result.

In [ ]:
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "05_tools.ipynb").exists() else ROOT / "notebooks" / "langchain"
LOG_FILE = NOTEBOOK_DIR / "04_tools" / "execution_log.txt"
execution_log = []  # The latest run is also available in this notebook variable.


def ask_with_manual_tools(question: str) -> str:
    execution_log.clear()

    def record(text: str, show: bool = True) -> None:
        execution_log.append(text)
        if show:
            print(text)

    messages = [
        SystemMessage(content="Use the provided tools for arithmetic. Give a short final answer."),
        HumanMessage(content=question),
    ]
    record(f"RUN: {datetime.now().astimezone().isoformat(timespec='seconds')}")
    record(f"MODEL: {model_name}")
    record("TOOL SCHEMAS GIVEN TO MODEL:\n" + json.dumps(TOOL_SCHEMAS, indent=2), show=False)
    record(f"USER: {question}")

    try:
        for step in range(1, 6):
            record("-" * 60)
            record(f"STEP {step}: MODEL INPUT")
            for message in messages:
                record(f"  {message.type}: content={message.content!r}")
                if getattr(message, "tool_calls", None):
                    record(f"    earlier tool_calls={message.tool_calls}")
                if message.type == "tool":
                    record(f"    tool_call_id={message.tool_call_id}")
            record(
                "FULL MODEL INPUT JSON:\n" + json.dumps(messages_to_dict(messages), indent=2, ensure_ascii=False),
                show=False,
            )

            reply = model_with_tools.invoke(messages)
            record(
                "FULL MODEL REPLY JSON:\n" + json.dumps(messages_to_dict([reply]), indent=2, ensure_ascii=False),
                show=False,
            )
            messages.append(reply)  # Keep the model's tool request in the conversation.

            if reply.invalid_tool_calls:
                raise ValueError(f"Invalid tool call: {reply.invalid_tool_calls}")

            if not reply.tool_calls:
                record(f"MODEL ANSWER (step {step}): {reply.content}")
                return reply.content

            for call in reply.tool_calls:
                name, args = call["name"], call["args"]
                record(f"MODEL REQUEST (step {step}): {name}({args}); call_id={call['id']}")
                if name not in TOOLS:
                    raise ValueError(f"Unknown tool: {name}")

                result = TOOLS[name](**args)
                record(f"PYTHON EXECUTES: {name}(**{args}) -> {result}")
                messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
                record(f"TOOL RESULT SENT TO MODEL: {result}")

        raise RuntimeError("The model did not finish within 5 steps.")
    except Exception as error:
        record(f"ERROR: {type(error).__name__}: {error}")
        raise
    finally:
        LOG_FILE.parent.mkdir(exist_ok=True)
        with LOG_FILE.open("a", encoding="utf-8") as output:
            output.write("\n".join(execution_log) + "\n\n")
        print(f"Saved execution log: {LOG_FILE}")

In [ ]:
ask_with_manual_tools("What is (7 + 5) * 3? Use the tools and show the result.")

### 2. LangChain agent with `@tool`

Now `@tool` builds the descriptions and argument schema from the functions. The next cell prints exactly what LangChain generated. `create_agent` performs the model/tool loop; `stream_mode="updates" shows the visible decisions, tool results, and final answer.

In [ ]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


native_tools = [add_numbers, multiply_numbers]
for selected_tool in native_tools:
    print(f"Python function: {selected_tool.name}{signature(selected_tool.func)}")
    print(f"Docstring read by @tool: {selected_tool.description}")
    print("LangChain input schema:")
    print(json.dumps(selected_tool.get_input_schema().model_json_schema(), indent=2))


agent = create_agent(
    model=model,
    tools=[add_numbers, multiply_numbers],
    system_prompt="Use the provided tools for arithmetic. Give a short final answer.",
)

In [ ]:
def ask_with_native_tools(question: str) -> None:
    print(f"USER: {question}")
    step = 0
    for update in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates",
    ):
        for node, state in update.items():
            step += 1
            print("-" * 60)
            print(f"STEP {step}: {node.upper()}")
            for message in state.get("messages", []):
                if message.type == "ai" and message.tool_calls:
                    for call in message.tool_calls:
                        print(f"MODEL DECISION: request {call['name']}({call['args']})")
                    if message.content:
                        print(f"MODEL CONTENT: {message.content}")
                elif message.type == "tool":
                    print(f"TOOL EXECUTED ({message.name}): {message.content}")
                elif message.type == "ai":
                    print(f"MODEL DECISION: final answer = {message.content}")
                else:
                    print(f"{node.upper()}: {message}")

In [ ]:
ask_with_native_tools("What is (7 + 5) * 3? Use the tools and show the result.")

In [ ]:
def short_ask_with_native_tools(question: str) -> str:
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    answer = result["messages"][-1].content

    print(f"USER: {question}")
    print(f"ANSWER: {answer}")

    return answer


short_ask_with_native_tools("What is (7 + 5) * 3? Use the tools and show the result.")

### 3. Tool with an explicit argument schema

Previously, `@tool` read types from the function. We can also provide a **custom Pydantic schema** via `args_schema`. The model receives field names and descriptions, and Python validates the values before the function is executed. In the example, we calculate a fictional delivery price; there is no shop or payment involved.

`QuoteInput` requires a distance from 1 to 100 km and allows selecting only `standard` or `express`. Notice that incorrect data does not pass validation.


In [ ]:
class QuoteInput(BaseModel):
    distance_km: int = Field(ge=1, le=100, description="Delivery distance in kilometers")
    speed: Literal["standard", "express"] = Field(
        default="standard", description="Delivery method"
    )


@tool(args_schema=QuoteInput)
def base_delivery_price(distance_km: int, speed: str = "standard") -> int:
    """Calculate a demonstration delivery price in PLN."""
    return 10 + 2 * distance_km + (8 if speed == "express" else 0)


print("Schema visible for the model:")
print(json.dumps(base_delivery_price.args_schema.model_json_schema(), ensure_ascii=False, indent=2))
print("Correct call via Python:", base_delivery_price.invoke({"distance_km": 5, "speed": "express"}))
try:
    base_delivery_price.invoke({"distance_km": -5, "speed": "rocket"})
except ValidationError as error:
    print("Incorrect arguments rejected:", error.error_count(), "errors")


The agent uses the same schema. The model may **request** the `base_delivery_price`, but it does not perform the calculation itself. In `messages`, we will see the model's request, the result of the Python function, and the model's response separately.


In [ ]:
schema_agent = create_agent(
    model=model,
    tools=[base_delivery_price],
    system_prompt="Always use base_delivery_price to estimate delivery costs. Answer briefly in English.",
)
schema_result = schema_agent.invoke({
    "messages": [{"role": "user", "content": "How much does standard delivery for 5 km cost?"}]
})
for message in schema_result["messages"]:
    if message.type == "ai" and message.tool_calls:
        print("MODEL REQUESTS:", message.tool_calls)
    elif message.type == "tool":
        print("PYTHON RETURNS:", message.content)
print("MODEL RESPONSE:", schema_result["messages"][-1].content)


### 4. The tool reads `context` and updates `state`

Now, the application will pass `QuoteContext(user_id, discount_percent)` when calling the agent. This context is **not visible as a tool argument to the model**. `ToolRuntime` provides it to the Python function. Imagine that the discount was previously determined by the application; in the notebook, we provide it manually as demo data.

The tool will save the calculated amount to `last_quote` in the **agent's state** via a `Command`. In that same `Command`, it will return a `ToolMessage` so the model can see the calculation result. `context` is not modified in this way.


In [ ]:
@dataclass(frozen=True)
class QuoteContext:
    user_id: str
    discount_percent: int


class QuoteState(AgentState):
    last_quote: int | None


TOOL_CONTEXT_SEEN = []  # Only a local didactic trace; we don't save anything.


@tool(args_schema=QuoteInput)
def quote_for_current_user(
    distance_km: int,
    runtime: ToolRuntime[QuoteContext, QuoteState],
    speed: str = "standard",
) -> Command:
    """Calculate a dummy delivery price with the current user's discount."""
    base = 10 + 2 * distance_km + (8 if speed == "express" else 0)
    discount = runtime.context.discount_percent
    price = base * (100 - discount) // 100
    TOOL_CONTEXT_SEEN.append((runtime.context.user_id, discount))
    return Command(update={
        "last_quote": price,
        "messages": [ToolMessage(
            content=f"Delivery price: {price} PLN.",
            tool_call_id=runtime.tool_call_id,
        )],
    })


print("Tool arguments visible to the model:", quote_for_current_user.args)


The application can provide a **new `context` object on the next call**. In the example, it changes the discount from 0% to 20% while keeping the `user_id`. The old context still has 0%; the tool does not mutate it during operation. Each call starts a separate run because we do not add a checkpointer here.


In [ ]:
context_agent = create_agent(
    model=model,
    tools=[quote_for_current_user],
    context_schema=QuoteContext,
    state_schema=QuoteState,
    system_prompt="Always use quote_for_current_user to estimate delivery. Answer briefly in English.",
)


def ask_for_quote(context: QuoteContext) -> dict:
    result = context_agent.invoke(
        {
            "messages": [{"role": "user", "content": "How much does standard delivery for 5 km cost?"}],
            "last_quote": None,
        },
        context=context,
    )
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("MODEL REQUESTS:", [call["name"] for call in message.tool_calls])
        elif message.type == "tool":
            print("PYTHON RETURNS:", message.content)
    print("STATE last_quote:", result["last_quote"])
    print("MODEL RESPONSE:", result["messages"][-1].content)
    return result


first_context = QuoteContext(user_id="u-101", discount_percent=0)
first_quote = ask_for_quote(first_context)
updated_context = replace(first_context, discount_percent=20)
second_quote = ask_for_quote(updated_context)
print("Old context:", first_context)
print("New context for next call:", updated_context)
print("ID and discount read exclusively by Python:", TOOL_CONTEXT_SEEN)


| Element | Source | Does the model provide it? | What changes? |
|---|---|---|---|
| `distance_km`, `speed` | tool request | yes, according to the schema | single tool call arguments |
| `user_id`, `discount_percent` | application `context` | no | the application creates a new context for the next `invoke` |
| `last_quote` | Python calculation result | no | the tool updates `state` via `Command` |
| response for the user | model after tool result | yes | final message |

**Most important:** `context` is the input for the current run. When a tool needs to save a result *during* a run, it updates `state`, not `runtime.context`. The code shows dummy quotes and a local trace; it does not call a database or an external system.
